In [1]:
import multiprocessing as mp
print("Number of processors: ", mp.cpu_count())

Number of processors:  32


In [8]:
import numpy as np
import timeit

# Prepare data
np.random.RandomState(100)
arr = np.random.randint(0, 10, size=[900000, 5])
data = arr.tolist()
data[:5]
len(data)

900000

In [9]:
# Solution Without Paralleization

def howmany_within_range(row, minimum, maximum):
    """Returns how many numbers lie within `maximum` and `minimum` in a given `row`"""
    count = 0
    for n in row:
        if minimum <= n <= maximum:
            count = count + 1
    return count

results = []

start_time = timeit.default_timer()
for row in data:
    results.append(howmany_within_range(row, minimum=4, maximum=8))
comp_time = timeit.default_timer() - start_time

print(comp_time)
print(results[:10])
#> [3, 1, 4, 4, 4, 2, 1, 1, 3, 3]

0.2379116070000009
[4, 3, 1, 5, 3, 0, 2, 2, 2, 3]


In [10]:
# Parallelizing using Pool.apply()

# Step 1: Init multiprocessing.Pool()
pool = mp.Pool(mp.cpu_count())
start_time = timeit.default_timer()
# Step 2: `pool.apply` the `howmany_within_range()`
results = [pool.apply(howmany_within_range, args=(row, 4, 8)) for row in data]
end_time = timeit.default_timer() - start_time
# Step 3: Don't forget to close
pool.close()

print(end_time)
print(results[:10])
#> [3, 1, 4, 4, 4, 2, 1, 1, 3, 3]

KeyboardInterrupt: 

In [20]:
# Parallelizing using Pool.map()

# Redefine, with only 1 mandatory argument.
def howmany_within_range_rowonly(row, minimum=4, maximum=8):
    count = 0
    for n in row:
        if minimum <= n <= maximum:
            count = count + 1
    return count

pool = mp.Pool(mp.cpu_count())
start_time = timeit.default_timer()

results = pool.map(howmany_within_range_rowonly, [row for row in data])

pool.close()
comp_time = timeit.default_timer() - start_time

print(comp_time)
print(results[:10])

0.16815326599999025
[4, 3, 1, 5, 3, 0, 2, 2, 2, 3]


In [26]:
# Parallelizing with Pool.starmap()
import multiprocessing as mp

pool = mp.Pool(mp.cpu_count())
start_time = timeit.default_timer()

results = pool.starmap(howmany_within_range, [(row, 4, 8) for row in data])

pool.close()
comp_time = timeit.default_timer() - start_time

print(comp_time)
print(results[:10])

0.5410498789999565
[4, 3, 1, 5, 3, 0, 2, 2, 2, 3]


In [27]:
# Parallel processing with Pool.apply_async()

pool = mp.Pool(mp.cpu_count())

results = []

# Step 1: Redefine, to accept `i`, the iteration number
def howmany_within_range2(i, row, minimum, maximum):
    """Returns how many numbers lie within `maximum` and `minimum` in a given `row`"""
    count = 0
    for n in row:
        if minimum <= n <= maximum:
            count = count + 1
    return (i, count)


# Step 2: Define callback function to collect the output in `results`
def collect_result(result):
    global results
    results.append(result)


# Step 3: Use loop to parallelize
for i, row in enumerate(data):
    pool.apply_async(howmany_within_range2, args=(i, row, 4, 8), callback=collect_result)

# Step 4: Close Pool and let all the processes complete    
pool.close()
pool.join()  # postpones the execution of next line of code until all processes in the queue are done.

# Step 5: Sort results [OPTIONAL]
results.sort(key=lambda x: x[0])
results_final = [r for i, r in results]

print(results_final[:10])
#> [3, 1, 4, 4, 4, 2, 1, 1, 3, 3]

Process ForkPoolWorker-545:
  File "/usr/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
Traceback (most recent call last):
  File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/lib/python3.10/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/usr/lib/python3.10/multiprocessing/queues.py", line 367, in get
    return _ForkingPickler.loads(res)
Process ForkPoolWorker-546:
AttributeError: Can't get attribute 'howmany_within_range2' on <module '__main__'>
Traceback (most recent call last):
  File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/lib/python3.10/multiproc

KeyboardInterrupt: 